# Day 7 Lab 09: Dimension Enrichment

        **Layer:** Silver  
        **Python reference:** `Lab_Files/labs/lab_09_dimension_enrichment.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Read current orders from Silver.
- Join customer, product, and FX dimensions through the shared helper.
- Inspect match status and normalized USD values.

**Dependency:** Run Lab/Notebook 08 first so `lake/silver/orders_current` exists.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
LAB_FILES = HERE / "Lab_Files"
if not LAB_FILES.exists():
    LAB_FILES = HERE.parent / "Lab_Files"

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")

## 1. Import Lab Helpers

In [ ]:
from day7_common import LAKE_DIR, OUTPUT_DIR, enriched_orders, ensure_output_dirs, require_source_data, spark_session, write_csv_dir

## 2. Start Spark and Read Current Orders

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook09DimensionEnrichment")

current = spark.read.parquet(str(LAKE_DIR / "silver" / "orders_current"))
current.select("order_id", "customer_id", "product_id", "currency", "amount").show(20, truncate=False)

## 3. Apply Dimension Enrichment

In [ ]:
enriched = enriched_orders(spark, current)

## 4. Inspect Enriched Rows

In [ ]:
preview = enriched.select(
    "order_id",
    "customer_name",
    "region",
    "product_name",
    "product_category",
    "status",
    "amount_usd",
    "customer_match_status",
).orderBy("order_id")
preview.show(20, truncate=False)

## 5. Write Enriched Silver Table

In [ ]:
enriched_path = LAKE_DIR / "silver" / "orders_enriched"
enriched.write.mode("overwrite").parquet(str(enriched_path))
write_csv_dir(preview, OUTPUT_DIR / "notebook_09_enriched_orders_preview")
print(f"Enriched current orders: {enriched.count()}")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()